# Tutorial 4: Secret Sharing with Polynomials


**Audience:** Completed Tutorial 1

**Prerequisites:** Scalar arithmetic

**Learning goals:**
- Implement Shamir's Secret Sharing step by step
- Reconstruct secrets via Lagrange interpolation
- Verify the threshold property


## What this notebook covers

1. Generating a secret-sharing polynomial
2. Evaluating the polynomial to create shares
3. Reconstructing the secret from any t shares
4. Verifying that fewer than t shares reveal nothing
5. Horner's method for efficient evaluation
6. Exercise: 3-of-5 sharing with exhaustive verification


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Scalar, Q
from frost.polynomial import generate_polynomial, evaluate_polynomial
from frost.lagrange import lagrange_coefficient


## Generate a Polynomial

In a t-of-n scheme, each participant generates a random polynomial of
degree t-1. The constant term (coefficient a₀) is the secret.


In [ ]:
threshold = 2  # 2-of-3 scheme
coefficients = generate_polynomial(threshold)
secret = coefficients[0]

print(f"Polynomial: f(x) = {int(secret)} + {int(coefficients[1])}·x")
print(f"Secret (a₀) = {int(secret)}")


## Evaluate at Participant Indexes

Each participant's share is the polynomial evaluated at their index.
For a 2-of-3 scheme, we evaluate f(1), f(2), and f(3).


In [ ]:
n_participants = 3
shares = []
for i in range(1, n_participants + 1):
    share = evaluate_polynomial(coefficients, i)
    shares.append(share)
    print(f"f({i}) = {int(share)}")


## Reconstruct from Any 2

Lagrange interpolation at x = 0 recovers the secret (the constant term).
Any 2 of the 3 shares are sufficient.


In [ ]:
# Use shares from participants 1 and 2
indexes = (1, 2)
reconstructed = Scalar(0)
for idx in indexes:
    lam = lagrange_coefficient(indexes, idx, 0)
    reconstructed = reconstructed + lam * shares[idx - 1]

print(f"Reconstructed secret: {int(reconstructed)}")
print(f"Original secret:      {int(secret)}")
print(f"Match? {int(reconstructed) == int(secret)}")


## Try a Different Pair

Any 2-of-3 combination recovers the same secret.


In [ ]:
for combo in [(1, 3), (2, 3)]:
    indexes = combo
    reconstructed = Scalar(0)
    for idx in indexes:
        lam = lagrange_coefficient(indexes, idx, 0)
        reconstructed = reconstructed + lam * shares[idx - 1]
    print(f"Shares {combo}: reconstructed = {int(reconstructed)}, match = {int(reconstructed) == int(secret)}")


## Insufficient Shares

With only 1 share, the secret is completely undetermined. A single point
on a line tells you nothing about where the line crosses x = 0.


In [ ]:
# With only 1 share, there are infinitely many degree-1 polynomials
# that pass through that point. The secret is uniformly distributed.
print("With 1 share, the secret could be anything.")
print(f"Share f(1) = {int(shares[0])}")
print("But f(0) is completely undetermined by a single point on a line.")


## Horner's Method

Naive polynomial evaluation requires t exponentiations. Horner's method
rewrites f(x) = a₀ + x·(a₁ + x·(a₂ + ...)), needing only t multiplications
and t additions. `evaluate_polynomial` uses Horner's method internally.


In [ ]:
# Manual Horner for f(x) = a₀ + a₁·x:
x = 2
horner = Scalar(0)
for coeff in reversed(coefficients):
    horner = Scalar(int(horner) * x + int(coeff))

direct = evaluate_polynomial(coefficients, x)
print(f"Horner:  f({x}) = {int(horner)}")
print(f"Library: f({x}) = {int(direct)}")
print(f"Match? {int(horner) == int(direct)}")


## Exercise

Generate a 3-of-5 sharing. Verify that ALL 3-element subsets of the
5 shares reconstruct the same secret.


In [ ]:
from itertools import combinations

coeffs_3of5 = generate_polynomial(3)
secret_3of5 = coeffs_3of5[0]
shares_3of5 = [evaluate_polynomial(coeffs_3of5, i) for i in range(1, 6)]

for combo in combinations(range(1, 6), 3):
    indexes = combo
    reconstructed = Scalar(0)
    for idx in indexes:
        lam = lagrange_coefficient(indexes, idx, 0)
        reconstructed = reconstructed + lam * shares_3of5[idx - 1]
    status = "✓" if int(reconstructed) == int(secret_3of5) else "✗"
    print(f"  {combo}: {status}")


## Pitfalls and Extensions

**Pitfall:** Shares must be kept secret. Publishing t shares reveals
the polynomial and the secret. In FROST, shares are never transmitted
in the clear after DKG.

**Extension:** Shamir's scheme requires a trusted dealer. FROST's DKG
eliminates the dealer by having every participant run their own Shamir's
scheme simultaneously. See Tutorial 5.
